# ENGR 5785G — Real-Time Stream Processing Assignment

## Scenario B: Hospital Patient Monitoring

It uses Spark Structured Streaming with:

- `readStream`
- watched-directory streaming
- `withWatermark`
- 2-minute tumbling window aggregation
- alert output for sustained elevated heart rate

**Important:** The dataset has heart-rate records but no real timestamp column.  
Therefore, the notebook uses the actual heart-rate values from the dataset and creates simulated event timestamps to stream the records.

In [1]:
# Install needed packages
# If Kaggle internet is off, PySpark/openpyxl are often already available.
!pip install pyspark openpyxl -q

In [2]:
from pathlib import Path
import os
import glob
import shutil
import time
import threading
import warnings

import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import col, window, avg, round as spark_round

warnings.filterwarnings("ignore")

print("Imports ready")

Imports ready


In [3]:
# Start Spark

spark = (
    SparkSession.builder
    .appName("ENGR5785G_IoMT_HeartRate_Streaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark started")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 20:46:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark started


## 1. Load actual Excel dataset

It searches `/kaggle/input` for `patients_data_with_alerts.xlsx`.

In [4]:
from pathlib import Path
import glob

INPUT_ROOT = Path("/kaggle/input")

# Preferred file name if available
preferred_name = "patients_data_with_alerts.xlsx"

# 1) Search for the preferred Excel file anywhere under /kaggle/input
matches = sorted(glob.glob(str(INPUT_ROOT / "**" / preferred_name), recursive=True))

# 2) If not found, search for any Excel file
if not matches:
    matches = sorted(glob.glob(str(INPUT_ROOT / "**" / "*.xlsx"), recursive=True))

# 3) If still not found, stop with clear error
if not matches:
    raise FileNotFoundError(
        "No Excel dataset found under /kaggle/input. "
        "Please add the dataset to this Kaggle notebook using Add Input."
    )

# Use the first matching file
dataset_path = Path(matches[0])

print("Dataset found:")
print(dataset_path)

raw_df = pd.read_excel(dataset_path, engine="openpyxl")

print("\nRaw dataset shape:", raw_df.shape)
print("\nColumns:")
print(list(raw_df.columns))

display(raw_df.head())

Dataset found:
/kaggle/input/datasets/mohamedahmed205/iomt-data/patients_data_with_alerts.xlsx

Raw dataset shape: (50000, 13)

Columns:
['Patient Number', 'Heart Rate (bpm)', 'SpO2 Level (%)', 'Systolic Blood Pressure (mmHg)', 'Diastolic Blood Pressure (mmHg)', 'Body Temperature (°C)', 'Fall Detection', 'Predicted Disease', 'Data Accuracy (%)', 'Heart Rate Alert', 'SpO2 Level Alert', 'Blood Pressure Alert', 'Temperature Alert']


,Patient Number,Heart Rate (bpm),SpO2 Level (%),Systolic Blood Pressure (mmHg),Diastolic Blood Pressure (mmHg),Body Temperature (°C),Fall Detection,Predicted Disease,Data Accuracy (%),Heart Rate Alert,SpO2 Level Alert,Blood Pressure Alert,Temperature Alert
0,1,96,92,101,89,36.904680,No,Diabetes Mellitus,95,Normal,Normal,Normal,Normal
1,2,76,83,103,85,37.254129,Yes,Asthma,91,Normal,Low,Normal,Abnormal
2,3,92,88,123,70,36.539418,Yes,Asthma,86,Normal,Low,Normal,Normal
3,4,137,84,167,97,37.018767,Yes,Asthma,99,High,Low,High,Normal
4,5,76,81,175,80,37.328671,No,Normal,93,Normal,Low,High,Abnormal


## 2. Prepare the streaming dataframe

dataset columns include:

- `Patient Number`
- `Heart Rate (bpm)`
- other health measurements and alert labels

Since the file is not a time-series log, this notebook:

- uses the actual `Heart Rate (bpm)` values
- maps rows into repeated patient streams, such as `P001`, `P002`, etc.
- creates event timestamps one second apart to simulate streaming arrival

In [5]:
# Required columns from dataset
HR_COL = "Heart Rate (bpm)"
PATIENT_NUMBER_COL = "Patient Number"

if HR_COL not in raw_df.columns:
    raise ValueError(f"Missing required column: {HR_COL}")

df = raw_df.copy()

# Keep only valid heart-rate values
df["heart_rate"] = pd.to_numeric(df[HR_COL], errors="coerce")
df = df.dropna(subset=["heart_rate"])
df = df[(df["heart_rate"] >= 30) & (df["heart_rate"] <= 250)]

# Use up to 50,000 rows as allowed by the assignment
df = df.head(50000).reset_index(drop=True)

# Simulate repeated patient streams.
# The original Patient Number is mostly one row per patient, so using it directly would not create
# consecutive windows for the same patient. This maps actual rows into 100 patient streams.
NUM_STREAM_PATIENTS = 100
df["patient_id"] = "P" + ((df.index % NUM_STREAM_PATIENTS) + 1).astype(str).str.zfill(3)

# Preserve original patient number if available
if PATIENT_NUMBER_COL in df.columns:
    df["source_patient_number"] = df[PATIENT_NUMBER_COL].astype(str)
else:
    df["source_patient_number"] = df.index.astype(str)

# Create event-time timestamps for streaming simulation
START_TIME = pd.Timestamp("2026-01-01 10:00:00")
df["event_time"] = START_TIME + pd.to_timedelta(df.index, unit="s")

stream_ready_df = df[["patient_id", "event_time", "heart_rate", "source_patient_number"]].copy()

print("Prepared streaming dataframe shape:", stream_ready_df.shape)
display(stream_ready_df.head(10))
display(stream_ready_df["heart_rate"].describe())

Prepared streaming dataframe shape: (50000, 4)


,patient_id,event_time,heart_rate,source_patient_number
0,P001,2026-01-01 10:00:00,96,1
1,P002,2026-01-01 10:00:01,76,2
2,P003,2026-01-01 10:00:02,92,3
3,P004,2026-01-01 10:00:03,137,4
4,P005,2026-01-01 10:00:04,76,5
5,P006,2026-01-01 10:00:05,123,6
6,P007,2026-01-01 10:00:06,61,7
7,P008,2026-01-01 10:00:07,103,8
8,P009,2026-01-01 10:00:08,111,9
9,P010,2026-01-01 10:00:09,146,10


count    50000.000000
mean       104.509420
std         25.922328
min         60.000000
25%         82.000000
50%        105.000000
75%        127.000000
max        149.000000
Name: heart_rate, dtype: float64

In [6]:
# Quick pre-check: estimate how many 2-minute high-HR windows exist before streaming

precheck = stream_ready_df.copy()
precheck["window_start"] = precheck["event_time"].dt.floor("2min")

pre_window_avg = (
    precheck
    .groupby(["patient_id", "window_start"], as_index=False)["heart_rate"]
    .mean()
    .rename(columns={"heart_rate": "avg_heart_rate"})
)

high_windows_count = (pre_window_avg["avg_heart_rate"] > 100).sum()

print("Number of high-HR patient windows where avg HR > 100:", int(high_windows_count))
display(pre_window_avg[pre_window_avg["avg_heart_rate"] > 100].head(10))

Number of high-HR patient windows where avg HR > 100: 23197


,patient_id,window_start,avg_heart_rate
0,P001,2026-01-01 10:00:00,111.5
2,P001,2026-01-01 10:04:00,124.0
6,P001,2026-01-01 10:12:00,105.0
9,P001,2026-01-01 10:18:00,125.0
10,P001,2026-01-01 10:20:00,122.0
11,P001,2026-01-01 10:22:00,135.0
12,P001,2026-01-01 10:24:00,102.0
14,P001,2026-01-01 10:28:00,110.0
17,P001,2026-01-01 10:34:00,111.0
18,P001,2026-01-01 10:36:00,103.0


## 3. Split actual dataset rows into streaming batch files

Spark Structured Streaming will watch a directory.  
The notebook copies batch files into that directory one by one to simulate real-time streaming.

In [7]:
BASE_DIR = Path("/kaggle/working/iomt_streaming_assignment")
SOURCE_DIR = BASE_DIR / "source_batches"
STREAM_DIR = BASE_DIR / "stream_input"
CHECKPOINT_DIR = BASE_DIR / "checkpoint_alerts"
ALERT_OUTPUT_PATH = BASE_DIR / "clinical_alerts.csv"

for d in [SOURCE_DIR, STREAM_DIR, CHECKPOINT_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

if ALERT_OUTPUT_PATH.exists():
    ALERT_OUTPUT_PATH.unlink()

# Write only the columns Spark will read
spark_input_df = stream_ready_df[["patient_id", "event_time", "heart_rate"]].copy()
spark_input_df["event_time"] = spark_input_df["event_time"].dt.strftime("%Y-%m-%d %H:%M:%S")

BATCH_SIZE = 1000

batch_count = 0
for start in range(0, len(spark_input_df), BATCH_SIZE):
    batch = spark_input_df.iloc[start:start + BATCH_SIZE]
    batch.to_csv(SOURCE_DIR / f"batch_{batch_count:04d}.csv", index=False)
    batch_count += 1

print(f"Created {batch_count} source batch files.")
print("Source batch folder:", SOURCE_DIR)
print("Watched streaming folder:", STREAM_DIR)

Created 50 source batch files.
Source batch folder: /kaggle/working/iomt_streaming_assignment/source_batches
Watched streaming folder: /kaggle/working/iomt_streaming_assignment/stream_input


## 4. Define Spark Structured Streaming pipeline

Assignment Scenario B logic:

- 2-minute tumbling window
- average heart rate per patient
- filter windows where average heart rate is above 100 bpm
- alert when the same patient has high heart rate in two consecutive windows

In [8]:
schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("heart_rate", DoubleType(), True)
])

stream_df = (
    spark.readStream
    .schema(schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .csv(str(STREAM_DIR))
)

windowed_avg = (
    stream_df
    .withWatermark("event_time", "5 minutes")
    .groupBy(
        "patient_id",
        window("event_time", "2 minutes")
    )
    .agg(
        spark_round(avg("heart_rate"), 2).alias("avg_heart_rate")
    )
)

high_hr_windows = windowed_avg.filter(col("avg_heart_rate") > 100)

print("Streaming pipeline defined successfully")

Streaming pipeline defined successfully


## 5. Alert function

This function checks whether the same patient exceeds 100 bpm in two consecutive 2-minute windows.

In [9]:
alert_state = {}
alert_records = []
alert_count = 0
MAX_ALERTS_TO_PRINT = 50

def detect_sustained_hr(batch_df, batch_id):
    global alert_state, alert_records, alert_count

    rows = (
        batch_df
        .select(
            col("patient_id"),
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            col("avg_heart_rate")
        )
        .orderBy("patient_id", "window_start")
        .collect()
    )

    if not rows:
        return

    print(f"\n================ Batch {batch_id} ================")

    for row in rows:
        patient_id = row["patient_id"]
        window_start = row["window_start"]
        window_end = row["window_end"]
        avg_hr = row["avg_heart_rate"]

        previous_high_window_end = alert_state.get(patient_id)

        if previous_high_window_end == window_start:
            alert_count += 1

            record = {
                "patient_id": patient_id,
                "avg_heart_rate": avg_hr,
                "window_start": str(window_start),
                "window_end": str(window_end),
                "alert": "Sustained elevated HR across 2 consecutive windows"
            }
            alert_records.append(record)

            if alert_count <= MAX_ALERTS_TO_PRINT:
                print(
                    "CLINICAL ALERT: Sustained elevated HR across 2 consecutive windows | "
                    f"patient_id={patient_id} | "
                    f"avg_hr={avg_hr} bpm | "
                    f"window_start={window_start} | "
                    f"window_end={window_end}"
                )
        else:
            # Print a small number of normal high-HR windows for evidence, but avoid too much spam.
            if alert_count < 5:
                print(
                    "High HR window detected | "
                    f"patient_id={patient_id} | "
                    f"avg_hr={avg_hr} bpm | "
                    f"window_start={window_start} | "
                    f"window_end={window_end}"
                )

        alert_state[patient_id] = window_end

## 6. Run the stream



In [10]:
# Stop previous active streams if you rerun this cell
for q in spark.streams.active:
    q.stop()

# Reset watched directory/checkpoint
for d in [STREAM_DIR, CHECKPOINT_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

alert_state.clear()
alert_records.clear()
alert_count = 0

query = (
    high_hr_windows
    .writeStream
    .outputMode("update")
    .foreachBatch(detect_sustained_hr)
    .option("checkpointLocation", str(CHECKPOINT_DIR))
    .start()
)

def feed_files(delay_seconds=1.5):
    files = sorted(SOURCE_DIR.glob("*.csv"))
    for f in files:
        shutil.copy(f, STREAM_DIR / f.name)
        print(f"Added file to stream: {f.name}")
        time.sleep(delay_seconds)

feeder = threading.Thread(target=feed_files, kwargs={"delay_seconds": 1.5})
feeder.start()

# Runtime is capped so the notebook does not run forever.
MAX_RUNTIME_SECONDS = min(180, max(70, batch_count * 2))

query.awaitTermination(MAX_RUNTIME_SECONDS)
query.stop()

print("\nStreaming finished")
print("Total clinical alerts fired:", alert_count)

if alert_records:
    alerts_df = pd.DataFrame(alert_records)
    alerts_df.to_csv(ALERT_OUTPUT_PATH, index=False)
    print("Saved alerts to:", ALERT_OUTPUT_PATH)
    display(alerts_df.head(20))
else:
    print("No clinical alerts fired. Try increasing MAX_RUNTIME_SECONDS or lowering BATCH_SIZE.")

Added file to stream: batch_0000.csv
Added file to stream: batch_0001.csv
Added file to stream: batch_0002.csv
Added file to stream: batch_0003.csv


Added file to stream: batch_0004.csv

================ Batch 0 ================
High HR window detected | patient_id=P001 | avg_hr=111.5 bpm | window_start=2026-01-01 10:00:00 | window_end=2026-01-01 10:02:00
High HR window detected | patient_id=P001 | avg_hr=124.0 bpm | window_start=2026-01-01 10:04:00 | window_end=2026-01-01 10:06:00
High HR window detected | patient_id=P001 | avg_hr=105.0 bpm | window_start=2026-01-01 10:12:00 | window_end=2026-01-01 10:14:00
High HR window detected | patient_id=P002 | avg_hr=133.0 bpm | window_start=2026-01-01 10:04:00 | window_end=2026-01-01 10:06:00
CLINICAL ALERT: Sustained elevated HR across 2 consecutive windows | patient_id=P002 | avg_hr=145.0 bpm | window_start=2026-01-01 10:06:00 | window_end=2026-01-01 10:08:00
CLINICAL ALERT: Sustained elevated HR across 2 consecutive windows | patient_id=P002 | avg_hr=105.0 bpm | window_start=2026-01-01 10:08:00 | window_end=2026-01-01 10:10:00
High HR window detected | patient_id=P002 | avg_hr=125.0 bpm

,patient_id,avg_heart_rate,window_start,window_end,alert
0,P002,145.0,2026-01-01 10:06:00,2026-01-01 10:08:00,Sustained elevated HR across 2 consecutive win...
1,P002,105.0,2026-01-01 10:08:00,2026-01-01 10:10:00,Sustained elevated HR across 2 consecutive win...
2,P003,113.0,2026-01-01 10:02:00,2026-01-01 10:04:00,Sustained elevated HR across 2 consecutive win...
3,P003,122.0,2026-01-01 10:10:00,2026-01-01 10:12:00,Sustained elevated HR across 2 consecutive win...
4,P003,116.0,2026-01-01 10:12:00,2026-01-01 10:14:00,Sustained elevated HR across 2 consecutive win...
5,P004,125.0,2026-01-01 10:02:00,2026-01-01 10:04:00,Sustained elevated HR across 2 consecutive win...
6,P004,112.0,2026-01-01 10:10:00,2026-01-01 10:12:00,Sustained elevated HR across 2 consecutive win...
7,P005,101.0,2026-01-01 10:06:00,2026-01-01 10:08:00,Sustained elevated HR across 2 consecutive win...
8,P005,101.0,2026-01-01 10:08:00,2026-01-01 10:10:00,Sustained elevated HR across 2 consecutive win...
9,P005,122.0,2026-01-01 10:10:00,2026-01-01 10:12:00,Sustained elevated HR across 2 consecutive win...


In [11]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

# Folder created by the notebook
folder_to_zip = Path("/kaggle/working/iomt_streaming_assignment")

# Output zip path
zip_base = Path("/kaggle/working/iomt_streaming_assignment_submission")
zip_path = str(zip_base) + ".zip"

# Remove old zip if it exists
if Path(zip_path).exists():
    Path(zip_path).unlink()

# Create zip
shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(folder_to_zip)
)

print("ZIP created:")
print(zip_path)

display(FileLink(zip_path))

ZIP created:
/kaggle/working/iomt_streaming_assignment_submission.zip


/kaggle/working/iomt_streaming_assignment_submission.zip